<a href="https://colab.research.google.com/github/omerlutfiduran/senior-design-project/blob/main/Senior_Desing_Project_Data_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import ee
ee.Authenticate()

True

In [7]:
ee.Initialize(project='senior-design-project-489117')


In [9]:
import ee
import folium

# Antalya sınırlarını tanımlama (FAO GAUL Level 1 veri setini kullanıyoruz)
antalya_fc = ee.FeatureCollection("FAO/GAUL/2015/level1") \
               .filter(ee.Filter.eq('ADM1_NAME', 'Antalya'))

# NASA FIRMS Aktif Yangın Veri Setini Çağırma
# Test için büyük Manavgat yangınlarının olduğu 2021 tarihini seçiyoruz.
start_date = '2021-07-01'
end_date = '2021-08-31'

firms_dataset = ee.ImageCollection('FIRMS') \
                  .filterBounds(antalya_fc) \
                  .filterDate(start_date, end_date)

# Sadece ısı anormalliği olan pikselleri (T21 bandı) seçip Antalya sınırına kırpıyoruz
fires = firms_dataset.select('T21').max().clip(antalya_fc)

# Harita Üzerinde Görselleştirme
# GEE görüntülerini Folium haritasına eklemek için yardımcı fonksiyon
def add_ee_layer(self, ee_image_object, vis_params, name):
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Map Data &copy; Google Earth Engine',
        name=name,
        overlay=True,
        control=True
    ).add_to(self)

folium.Map.add_ee_layer = add_ee_layer

# Antalya merkezli interaktif bir harita oluşturalım
my_map = folium.Map(location=[36.9, 30.7], zoom_start=8)

# Antalya sınırını haritaya ekleme
empty = ee.Image().byte()
outline = empty.paint(featureCollection=antalya_fc, color=1, width=2)
my_map.add_ee_layer(outline, {'palette': ['000000']}, 'Antalya Siniri')

# Yangın noktalarını haritaya ekleme (Isı şiddetine göre renk gradyanı)
# 300 Kelvin (sarı) ile 500 Kelvin (koyu kırmızı) arasında doğrusal bir renk geçişi sağlanır.
fire_vis_params = {
    'min': 300,
    'max': 500,
    'palette': ['yellow', 'orange', 'red', 'darkred']
}
my_map.add_ee_layer(fires, fire_vis_params, '2021 Yanginlari (FIRMS - Isi Siddeti)')

# Harita katman kontrolünü ekle ve göster
my_map.add_child(folium.LayerControl())
display(my_map)